# ?? SignalScope: Master Dual-Stream ConvNeXt-Tiny + SRM Forensic Model Trainer
**Smart India Hackathon (SIH 2026) | Problem Statement 2**
Domain: **AI Media Forensics / Trust & Safety**

### ?? Primary Objective: Generalization to Unseen AI Generators
- **Semantic Stream**: ConvNeXt-Tiny (Pretrained)
- **Forensic Stream**: Spatial Rich Model (SRM) High-Pass Noise Residuals
- **Training Strategy**: Mixed-Precision (AMP) Two-Phase Differential Fine-Tuning
- **Evaluation Split**: Held-Out Midjourney & VQDM (Zero exposure during training)

> ?? **IMPORTANT**: Pehle menu mein `Runtime` -> `Change runtime type` -> select **T4 GPU** karein.

## 1. Verify GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("?? Warning: GPU not detected! Please set Runtime -> Change runtime type -> T4 GPU.")

## 2. Mount Google Drive & Install Required Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm pyyaml matplotlib scikit-learn

## 3. Clone Repository & Setup Working Directory

In [ ]:
import os
if not os.path.exists('/content/sih_1'):
    !git clone https://github.com/vishvjani/sih_1.git /content/sih_1
else:
    %cd /content/sih_1
    !git pull origin main

%cd /content/sih_1
import sys
if '/content/sih_1/model_engine' not in sys.path:
    sys.path.insert(0, '/content/sih_1/model_engine')
print('Current Working Directory:', os.getcwd())

## 4. Dataset Setup
Agar aapne GenImage ki `.zip` file apne Google Drive mein download ki hai, toh niche diye cell se unzip karein. Agar koi zip nahi hai, toh yeh cell automatically sample dataset taiyar kar dega taaki training turant chal sake.

In [ ]:
import os
from pathlib import Path
from PIL import Image
import numpy as np

data_root = Path('/content/data/GenImage')
data_root.mkdir(parents=True, exist_ok=True)

# Check if user has zip in Google Drive
drive_zips = list(Path('/content/drive/MyDrive').glob('*GenImage*.zip')) + list(Path('/content/drive/MyDrive').glob('*Diffusion*.zip'))
if drive_zips:
    print(f"Found dataset zip in Drive: {drive_zips[0]}. Unzipping...")
    !unzip -q "{drive_zips[0]}" -d /content/data/GenImage/
else:
    print("No Drive zip found. Generating structured sample dataset so training runs out-of-the-box...")
    for gen in ['stable_diffusion_v1_4', 'glide', 'wukong', 'biggan', 'stable_diffusion_v1_5', 'midjourney', 'vqdm']:
        for split in ['train', 'val']:
            (data_root / gen / split / 'nature').mkdir(parents=True, exist_ok=True)
            (data_root / gen / split / 'ai').mkdir(parents=True, exist_ok=True)
            for i in range(15):
                real_img = Image.fromarray(np.uint8(np.random.rand(256, 256, 3) * 255))
                ai_img = Image.fromarray(np.uint8(np.random.rand(256, 256, 3) * 255))
                real_img.save(data_root / gen / split / 'nature' / f'real_{i}.jpg')
                ai_img.save(data_root / gen / split / 'ai' / f'ai_{i}.jpg')

print("? Dataset folder structure verified at:", data_root)

## 5. Phase 0: Dataset Audit & Anti-Leakage Manifest Creation

In [ ]:
from src.data.audit import DatasetAuditor
from src.data.split import GeneratorSplitter

output_dir = Path('/content/sih_1/manifests')
output_dir.mkdir(parents=True, exist_ok=True)

auditor = DatasetAuditor(data_root)
gen_folders = [p for p in data_root.iterdir() if p.is_dir()]
unique_reals_dict, dupes = auditor.find_real_image_duplicates(gen_folders)
unique_reals = list(unique_reals_dict.values())

# Collect AI paths by generator
ai_by_gen = {}
for gen_dir in gen_folders:
    ai_imgs = list((gen_dir / 'train' / 'ai').glob('*.*')) + list((gen_dir / 'val' / 'ai').glob('*.*'))
    ai_by_gen[gen_dir.name] = ai_imgs

print(f"Found {len(unique_reals)} unique real images and {sum(len(v) for v in ai_by_gen.values())} AI images across {len(ai_by_gen)} generators.")

splitter = GeneratorSplitter(seed=42)
manifests = splitter.create_100k_manifests(unique_reals, ai_by_gen, output_dir)
print("? Manifests created successfully:", manifests)

## 6. Initialize Dual-Stream Architecture (ConvNeXt-Tiny + SRM)

In [ ]:
from src.models.network import DualStreamSignalScope

device = "cuda" if torch.cuda.is_available() else "cpu"
model = DualStreamSignalScope(pretrained=True, dropout_rate=0.3, use_srm_stream=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"? Model Initialized on {device}")
print(f"Total Parameters: {total_params / 1e6:.2f}M | Initial Trainable: {trainable_params / 1e6:.2f}M")

## 7. Two-Phase Differential Training (Mixed Precision AMP)
- **Phase 1**: Backbone Frozen, Classifier Warmup
- **Phase 2**: Stage 3 & 4 Differential Fine-Tuning
- Automatically saves `signalscope_final_calibrated.pth` in `/content/sih_1/checkpoints/`

In [ ]:
from src.preprocessing.transforms import get_training_transforms, get_inference_transforms
from src.data.dataset import GenImageDataset
from src.training.trainer import SignalScopeTrainer
from torch.utils.data import DataLoader

train_manifest = '/content/sih_1/manifests/train_manifest.json'
val_manifest = '/content/sih_1/manifests/val_manifest.json'
test_manifest = '/content/sih_1/manifests/test_unseen_manifest.json'

train_ds = GenImageDataset(train_manifest, transform=get_training_transforms(256))
val_ds = GenImageDataset(val_manifest, transform=get_inference_transforms(256))
test_ds = GenImageDataset(test_manifest, transform=get_inference_transforms(256))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

print(f"Ready to train: {len(train_ds)} train samples, {len(val_ds)} val samples, {len(test_ds)} unseen test samples.")

trainer = SignalScopeTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_unseen_loader=test_loader,
    device=device,
    checkpoint_dir='/content/sih_1/checkpoints'
)

# Start training (Phase 1: 3 epochs warmup, Phase 2: 7 epochs fine-tuning)
training_history = trainer.train(phase1_epochs=3, phase2_epochs=7)

## 8. Benchmark Evaluation on Unseen Generators
Evaluates model on held-out Midjourney and VQDM images.

In [ ]:
metrics, logits, labels = trainer.evaluate(test_loader)
print("\n=============================================")
print("?? UNSEEN GENERATOR EVALUATION BENCHMARK")
print("=============================================")
print(f"Overall ROC-AUC:          {metrics['overall_roc_auc']:.4f}")
print(f"Unseen-Gen ROC-AUC:       {metrics['unseen_generator_roc_auc']:.4f}")
print(f"Macro-F1 Score:           {metrics['macro_f1']:.4f}")
print(f"False Positive Rate (FPR): {metrics['false_positive_rate']:.4f}")
print("Confusion Matrix:", metrics['confusion_matrix'])

## 9. Grad-CAM Localized Visual Explanations
Generates visual attribution heatmap for an unseen test sample.

In [ ]:
from src.explainability.gradcam import GradCAM
import matplotlib.pyplot as plt

gradcam = GradCAM(model)
sample_batch = next(iter(test_loader))
sample_img_t = sample_batch['image'][0:1].to(device)

heatmap = gradcam.generate_heatmap(sample_img_t)

plt.figure(figsize=(6, 6))
plt.title("SignalScope Localized Grad-CAM Heatmap")
plt.imshow(heatmap, cmap='jet')
plt.axis('off')
plt.show()
print("? Grad-CAM heatmap generated successfully!")

## 10. Export Calibrated Weights Directly to Google Drive

In [ ]:
import shutil

drive_export_dir = Path('/content/drive/MyDrive/SignalScope_Checkpoints')
drive_export_dir.mkdir(parents=True, exist_ok=True)

ckpt_src = Path('/content/sih_1/checkpoints/signalscope_final_calibrated.pth')
if ckpt_src.exists():
    dest = drive_export_dir / 'signalscope_final_calibrated.pth'
    shutil.copy(ckpt_src, dest)
    print(f"?? SUCCESS! Model weights exported to Google Drive: {dest}")
    print(f"File size: {dest.stat().st_size / (1024 * 1024):.2f} MB")
else:
    print("? Checkpoint not found. Make sure Cell 7 (Training) has run and completed.")